In [1]:
import time
import pandas as pd
import requests
from pathlib import Path
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

DATA_DIR = Path.cwd() / 'data'
if not DATA_DIR.exists():
    DATA_DIR = Path.cwd().parent / 'data'


In [2]:
# Use merged_geo.csv to determine whether a CCD high school is in our network
full_merged_data = pd.read_csv(DATA_DIR / 'merged_geo.csv')
full_merged_data['hs_id'] = full_merged_data['hs_id'].astype('string').str.strip()

network_public_hs_ids = set(
    full_merged_data.loc[full_merged_data['school_type'] == 'public', 'hs_id']
        .dropna()
        .unique()
        .tolist()
)
len(network_public_hs_ids)

80285

In [3]:
# Pull school-level enrollment by race for 2019 and 2023 (including race=99 total)
session = requests.Session()
retries = Retry(
    total=8,
    connect=8,
    read=8,
    status=8,
    backoff_factor=0.75,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(["GET"]),
    respect_retry_after_header=True,
)
session.mount("https://", HTTPAdapter(max_retries=retries, pool_connections=10, pool_maxsize=10))
session.headers.update({"User-Agent": "c2i-ccd-enrollment-race/1.0"})

def fetch_json(url: str) -> dict:
    for attempt in range(6):
        try:
            resp = session.get(url, timeout=(15, 120))
            resp.raise_for_status()
            return resp.json()
        except (requests.exceptions.ChunkedEncodingError, requests.exceptions.ConnectionError):
            if attempt == 5:
                raise
            time.sleep(1.5 * (attempt + 1))
    raise RuntimeError("Unreachable")

years = [2019, 2023]
race_codes = [1, 2, 3, 4, 5, 6, 7, 99]
all_school_enroll = []

for year in years:
    for race in race_codes:
        url = f"https://educationdata.urban.org/api/v1/schools/ccd/enrollment/{year}/grade-99/race/?race={race}&sex=99"
        while url:
            data = fetch_json(url)
            results = data.get('results', [])
            for r in results:
                r.setdefault('year', year)
                r.setdefault('race', race)
            all_school_enroll.extend(results)
            url = data.get('next')
            time.sleep(0.05)
        print(f"Finished year {year}, race {race}, total rows: {len(all_school_enroll):,}")

school_enroll_df = pd.DataFrame(all_school_enroll)
print('school_enroll_df shape:', school_enroll_df.shape)
school_enroll_df.head()

Finished year 2019, race 1, total rows: 99,290
Finished year 2019, race 2, total rows: 198,580
Finished year 2019, race 3, total rows: 297,870
Finished year 2019, race 4, total rows: 397,160
Finished year 2019, race 5, total rows: 496,450
Finished year 2019, race 6, total rows: 595,740
Finished year 2019, race 7, total rows: 695,030
Finished year 2019, race 99, total rows: 794,320
Finished year 2023, race 1, total rows: 893,998
Finished year 2023, race 2, total rows: 993,676
Finished year 2023, race 3, total rows: 1,093,354
Finished year 2023, race 4, total rows: 1,193,032
Finished year 2023, race 5, total rows: 1,292,710
Finished year 2023, race 6, total rows: 1,392,388
Finished year 2023, race 7, total rows: 1,492,066
Finished year 2023, race 99, total rows: 1,591,744
school_enroll_df shape: (1591744, 9)


,year,ncessch,ncessch_num,grade,race,sex,enrollment,fips,leaid
0,2019,010000500870,10000500870,99,1,99,351.0,1,100005
1,2019,010000500871,10000500871,99,1,99,711.0,1,100005
2,2019,010000500879,10000500879,99,1,99,378.0,1,100005
3,2019,010000500889,10000500889,99,1,99,352.0,1,100005
4,2019,010000501616,10000501616,99,1,99,230.0,1,100005


In [ ]:
# Pivot + calculate race percentages using race=99 as total
for col in ['ncessch', 'year', 'race', 'enrollment']:
    if col not in school_enroll_df.columns:
        raise ValueError(f"Missing expected column: {col}")

school_enroll_df['ncessch'] = school_enroll_df['ncessch'].astype('string').str.strip()
school_enroll_df['year'] = pd.to_numeric(school_enroll_df['year'], errors='coerce')
school_enroll_df['race'] = pd.to_numeric(school_enroll_df['race'], errors='coerce')
school_enroll_df['enrollment'] = pd.to_numeric(school_enroll_df['enrollment'], errors='coerce')

school_pivot = (
    school_enroll_df.pivot_table(
        index=['ncessch', 'year'],
        columns='race',
        values='enrollment',
        aggfunc='sum'
    )
    .reset_index()
)

total = school_pivot.get(99)
if total is None:
    raise ValueError('Race code 99 (total) not present in pulled data')

school_pivot['total'] = total
school_pivot.loc[school_pivot['total'].isna() | (school_pivot['total'] == 0), 'total'] = pd.NA

school_pivot['hs_pct_white'] = school_pivot.get(1, 0) / school_pivot['total']
school_pivot['hs_pct_black'] = school_pivot.get(2, 0) / school_pivot['total']
school_pivot['hs_pct_hispanic'] = school_pivot.get(3, 0) / school_pivot['total']
school_pivot['hs_pct_asian'] = school_pivot.get(4, 0) / school_pivot['total']
school_pivot['hs_pct_aian'] = school_pivot.get(5, 0) / school_pivot['total']
school_pivot['hs_pct_nhpi'] = school_pivot.get(6, 0) / school_pivot['total']
school_pivot['hs_pct_two_or_more'] = school_pivot.get(7, 0) / school_pivot['total']

hs_race = school_pivot[[
    'year',
    'ncessch',
    'hs_pct_white',
    'hs_pct_black',
    'hs_pct_hispanic',
    'hs_pct_asian',
    'hs_pct_aian',
    'hs_pct_nhpi',
    'hs_pct_two_or_more',
]].copy()

hs_race.rename(columns={'ncessch': 'hs_id'}, inplace=True)
hs_race['hs_id'] = hs_race['hs_id'].astype('string').str.strip()
hs_race['matched'] = hs_race['hs_id'].isin(network_public_hs_ids).astype('int64')

hs_race.sort_values(['year', 'hs_id'], inplace=True)
hs_race.reset_index(drop=True, inplace=True)
hs_race.head()

race,year,hs_id,hs_pct_white,hs_pct_black,hs_pct_hispanic,hs_pct_asian,hs_pct_aian,hs_pct_nhpi,hs_pct_two_or_more,matched
0,2019,010000500870,0.407666,0.032520,0.528455,0.003484,0.001161,0.000000,0.026713,1
1,2019,010000500871,0.457529,0.039254,0.471042,0.005148,0.001931,0.000000,0.025097,1
2,2019,010000500879,0.416300,0.033040,0.503304,0.003304,0.004405,0.001101,0.038546,1
3,2019,010000500889,0.376874,0.031049,0.559957,0.003212,0.004283,0.000000,0.024625,1
4,2019,010000501616,0.378913,0.049423,0.535420,0.004942,0.004942,0.000000,0.026359,1


In [ ]:
# should be > 0 (multiple races per school-year)
school_enroll_df.duplicated(["ncessch", "year"]).sum()

# should be 0 (one row per school-year)
hs_race.duplicated(["hs_id", "year"]).sum()

# confirm columns (should NOT include 'race')
hs_race.columns


1392776
0
Index(['year', 'hs_id', 'hs_pct_white', 'hs_pct_black', 'hs_pct_hispanic',
       'hs_pct_asian', 'hs_pct_aian', 'hs_pct_nhpi', 'hs_pct_two_or_more',
       'matched'],
      dtype='str', name='race')


In [8]:
hs_race.columns.name = None        # right before displaying hs_race


In [9]:
hs_race.to_csv(DATA_DIR / 'hs_race.csv', index=False)
hs_race.head()

,year,hs_id,hs_pct_white,hs_pct_black,hs_pct_hispanic,hs_pct_asian,hs_pct_aian,hs_pct_nhpi,hs_pct_two_or_more,matched
0,2019,010000500870,0.407666,0.032520,0.528455,0.003484,0.001161,0.000000,0.026713,1
1,2019,010000500871,0.457529,0.039254,0.471042,0.005148,0.001931,0.000000,0.025097,1
2,2019,010000500879,0.416300,0.033040,0.503304,0.003304,0.004405,0.001101,0.038546,1
3,2019,010000500889,0.376874,0.031049,0.559957,0.003212,0.004283,0.000000,0.024625,1
4,2019,010000501616,0.378913,0.049423,0.535420,0.004942,0.004942,0.000000,0.026359,1


In [12]:
# Compare like-for-like: unique public HS in network (2019 & 2023) vs unique matched HS in hs_race (2019 & 2023)

years = [2019, 2023]

# IMPORTANT: `hs_race['matched']` was computed against *all* public hs_id in merged_geo.csv,
# not year-specific. For a like-for-like comparison by year, compute intersection with
# public HS present in merged_geo.csv for those same cycles.

hs_race_years = hs_race.loc[hs_race['year'].isin(years), ['hs_id', 'year']].copy()
hs_race_years['hs_id'] = hs_race_years['hs_id'].astype('string').str.strip()

network_years = full_merged_data.loc[
    (full_merged_data['school_type'] == 'public') & (full_merged_data['cycle'].isin(years)),
    ['hs_id', 'cycle']
].copy()
network_years['hs_id'] = network_years['hs_id'].astype('string').str.strip()
network_years.rename(columns={'cycle': 'year'}, inplace=True)

# Unique HS ids in network for these years
network_unique_public_hs = network_years['hs_id'].dropna().nunique()

# Unique HS ids in CCD race pull (regardless of network)
ccd_unique_hs = hs_race_years['hs_id'].dropna().nunique()

# Unique HS ids that are present in BOTH datasets (same hs_id + year)
matched_keys = hs_race_years.merge(network_years.drop_duplicates(), on=['hs_id', 'year'], how='inner')
matched_unique_hs = matched_keys['hs_id'].dropna().nunique()

print('Years:', years)
print('Unique public hs_id in merged_geo.csv (these years):', network_unique_public_hs)
print('Unique hs_id in hs_race (these years):', ccd_unique_hs)
print('Unique hs_id in intersection (hs_id+year):', matched_unique_hs)

# Optional: quick diagnostics on which ids are missing where (ignores year)
network_ids = set(network_years['hs_id'].dropna().unique().tolist())
ccd_ids = set(hs_race_years['hs_id'].dropna().unique().tolist())
print('Network ids not in CCD pull:', len(network_ids - ccd_ids))
print('CCD ids not in network:', len(ccd_ids - network_ids))


Years: [2019, 2023]
Unique public hs_id in merged_geo.csv (these years): 71039
Unique hs_id in hs_race (these years): 103565
Unique hs_id in intersection (hs_id+year): 69627
Network ids not in CCD pull: 986
CCD ids not in network: 33512
